# AgentCore Runtime 외부에서 호스팅되는 LlamaIndex를 위한 Amazon Bedrock AgentCore Observability

이 Notebook에서는 Amazon Bedrock AgentCore Runtime 외부에서 호스팅되는 [Llama Index Agent](https://docs.llamaindex.ai/en/stable/use_cases/agents/)에 관찰성을 설정하는 방법을 살펴봅니다. 설정을 완료하면 Amazon CloudWatch의 GenAI Observability dashboard에서 LlamaIndex 에이전트의 내부 의사 결정 과정을 확인할 수 있습니다.

## 학습 내용
- Amazon OpenTelemetry Python Instrumentation으로 LlamaIndex 에이전트를 설정하는 방법
- Amazon CloudWatch GenAI Observability에서 에이전트 trace를 시각화하고 분석하는 방법


## 사전 요구 사항
- Amazon CloudWatch에서 Transaction Search를 활성화합니다. 처음 사용하는 경우 Bedrock AgentCore span과 trace를 확인하려면 CloudWatch Transaction Search를 활성화해야 합니다. 자세한 내용은 [문서](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html)를 참고하세요.
- environment variable에 추가할 Amazon CloudWatch log group 및 log stream을 구성합니다.
- 모델 ID가 global.anthropic.claude-haiku-4-5-20251001-v1:0인 Claude Haiku 4.5에 대한 Amazon Bedrock 모델 액세스 권한이 있는 AWS 계정
- `aws configure`를 사용하여 구성한 AWS credentials
- environment variable을 반영한 .env 파일. `.env.example`에 예제가 제공됩니다.

## 1. 설정 및 설치

이 Notebook을 실행하기 전에 다음 단계에 따라 virtual environment를 설정했는지 확인하세요.

1. Terminal에서 LlamaIndex 디렉터리로 이동합니다.

2. virtual environment를 생성하고 활성화합니다.
   ```bash
   # virtual environment 생성
   python -m venv venv

   # virtual environment 활성화
   # Windows
   venv\Scripts\activate
   # macOS/Linux
   source venv/bin/activate
   ```

3. dependency를 설치합니다.
   ```bash
   pip install -r requirements.txt
   ```

4. Jupyter 또는 VS Code에서 Notebook을 열 때 다음을 수행합니다.
   - kernel selector에서 "venv" kernel을 선택합니다.
   - kernel이 목록에 나타나지 않으면 Jupyter 또는 VS Code를 다시 시작합니다.

#### 사전 요구 리소스 배포

시작하기 전에 AgentCore Observability를 위한 log group과 log stream을 생성합니다.

In [ ]:
import boto3

cloudwatch_client = boto3.client("logs", region_name="us-west-2")
response = cloudwatch_client.create_log_group(
    logGroupName="agents/llama-index-agent-logs",
)
response

In [ ]:
response = cloudwatch_client.create_log_stream(logGroupName="agents/llama-index-agent-logs", logStreamName="default")
response

#### Transaction Search 활성화

이 예제를 실행하려면 먼저 Transaction Search를 활성화해야 합니다. 이 [링크](https://console.aws.amazon.com/cloudwatch/home#xray:settings/transaction-search)를 통해 AWS console에서 활성화할 수 있습니다.

페이지에서 edit을 클릭한 다음 span을 OpenTelemetry 형식의 structured log로 수집하도록 옵션을 설정합니다.

![image.png](./images/transactional_search.png)
![image.png](./images/transactional_search2.png)

## 2. 환경 구성
LlamaIndex 에이전트의 관찰성을 활성화하고 telemetry 데이터를 Amazon CloudWatch로 전송하려면 다음 environment variable을 구성해야 합니다. `.env` 파일을 사용하면 민감한 AWS credentials를 코드와 분리하여 안전하게 관리하면서도 환경 간에 쉽게 전환할 수 있습니다.

**AWS credentials가 구성되어 있는지 확인하세요.**

environment variable 구성을 위한 `.env` 파일을 생성합니다. `env.example`을 template으로 사용하세요.

필수 Environment Variable:

| Variable | 값 | 용도 |
|----------|-------|---------|
| `OTEL_PYTHON_DISTRO` | `aws_distro` | AWS Distro for OpenTelemetry(ADOT) 사용 |
| `OTEL_PYTHON_CONFIGURATOR` | `aws_configurator` | ADOT SDK용 AWS configurator 설정 |
| `OTEL_EXPORTER_OTLP_PROTOCOL` | `http/protobuf` | export protocol 구성 |
| `OTEL_EXPORTER_OTLP_LOGS_HEADERS` | `x-aws-log-group=<YOUR-LOG-GROUP>,x-aws-log-stream=<YOUR-LOG-STREAM>,x-aws-metric-namespace=<YOUR-NAMESPACE>` | log를 CloudWatch group으로 전송 |
| `OTEL_RESOURCE_ATTRIBUTES` | `service.name=<YOUR-AGENT-NAME>` | 관찰성 데이터에서 에이전트 식별 |
| `AGENT_OBSERVABILITY_ENABLED` | `true` | ADOT pipeline 활성화 |
| `AWS_REGION` | `<YOUR-REGION>` | AWS Region |

In [ ]:
%%writefile .env
# AWS OpenTelemetry 배포판
OTEL_PYTHON_DISTRO=aws_distro
OTEL_PYTHON_CONFIGURATOR=aws_configurator

# 내보내기 protocol
OTEL_EXPORTER_OTLP_PROTOCOL=http/protobuf
OTEL_TRACES_EXPORTER=otlp

# CloudWatch 통합(필요에 따라 주석을 해제하고 구성)
OTEL_EXPORTER_OTLP_LOGS_HEADERS=x-aws-log-group=agents/llama-index-agent-logs10,x-aws-log-stream=default,x-aws-metric-namespace=bedrock-agentcore

# 서비스 식별
OTEL_RESOURCE_ATTRIBUTES=service.name=agentic-llamaindex-agentcore
# Agent Observability 활성화
AGENT_OBSERVABILITY_ENABLED=true

# span noise를 줄이기 위해 계측 비활성화(선택 사항)
OTEL_PYTHON_DISABLED_INSTRUMENTATIONS=jinja2

## 3. Environment Variable 불러오기

`.env` 파일에서 environment variable을 불러옵니다.

In [ ]:
import os
from dotenv import load_dotenv

# .env 파일에서 environment variable 불러오기
load_dotenv()

# OTEL 관련 environment variable 표시
otel_vars = [
    "OTEL_PYTHON_DISTRO",
    "OTEL_PYTHON_CONFIGURATOR",
    "OTEL_EXPORTER_OTLP_PROTOCOL",
    "OTEL_EXPORTER_OTLP_LOGS_HEADERS",
    "OTEL_RESOURCE_ATTRIBUTES",
    "AGENT_OBSERVABILITY_ENABLED",
    "OTEL_TRACES_EXPORTER",
    "OTEL_PYTHON_DISABLED_INSTRUMENTATIONS",
]

print("OpenTelemetry Configuration:")
for var in otel_vars:
    value = os.getenv(var)
    if value:
        print(f"{var}={value}")

## 4. Python 파일에 LlamaIndex Agent 생성

LlamaIndex 산술 에이전트 구현은 `llama_index_agent.py`에 제공됩니다. 이 에이전트는 Amazon Bedrock의 Claude 3 Haiku 모델을 사용하도록 설정된 간단한 산술 에이전트입니다. `opentelemetry-instrument` 명령을 사용하면 AWS OpenTelemetry 배포판에서 tracer provider 설정을 자동으로 처리합니다.

이 에이전트는 다음 작업을 수행하는 간단한 산술 에이전트입니다.

- AWS Bedrock의 Claude Haiku 모델을 사용하는 FunctionAgent 생성
- 덧셈과 곱셈을 위한 기본 산술 tool 정의
- 간단한 수식 (121 + 2) * 5를 계산하는 task를 에이전트에 부여
- 에이전트를 실행하고 계산 결과 반환

에이전트는 다음과 같이 구성됩니다.

- 두 개의 산술 함수 tool: add 및 multiply
- Large Language Model로 사용하는 Amazon Bedrock의 Claude Haiku 모델
- tracing 및 관찰성을 위한 OpenTelemetry 계측

에이전트는 수학 query를 처리하고 결과를 반환하는 run method를 사용하여 비동기식으로 실행됩니다.

In [ ]:
%%writefile llama_index_agent.py
###########################
#### 아래는 Agent 코드입니다. ####
###########################
import os
import asyncio
import logging
from llama_index.observability.otel import LlamaIndexOpenTelemetry
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent

# LlamaIndex용 OpenTelemetry 계측 초기화
instrumentor = LlamaIndexOpenTelemetry(debug=True)

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# LlamaIndex 로깅 구성
logging.getLogger("llamaindex").setLevel(logging.INFO)

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b


def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b


def get_bedrock_model():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")
    
    try:
        # boto3가 credential resolution을 자동으로 처리하도록 설정
        bedrock_model = BedrockConverse(
            model=model_id,
            region_name=region,
            # credential을 명시하지 않아도 boto3가 자동으로 검색
        )
        logger.info(f"Successfully initialized Bedrock model: {model_id} in region: {region}")
        return bedrock_model
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock model: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

# 모델 초기화
bedrock_model = get_bedrock_model()

# 산술 에이전트 생성
agent = FunctionAgent(
    tools=[add, multiply],
    llm=bedrock_model,
)

# 수신 시작
instrumentor.start_registering()

# 산술 task 실행
query = """What is (121 + 2) * 5?"""

async def main():
    result = await agent.run(query)
    print("Result:", str(result))

asyncio.run(main())


## 5. AWS OpenTelemetry Python Distro

환경 구성을 마쳤으므로 관찰성이 작동하는 방식을 살펴보겠습니다. [AWS OpenTelemetry Python Distro](https://pypi.org/project/aws-opentelemetry-distro/)는 코드를 변경하지 않고도 telemetry 데이터를 수집할 수 있도록 LlamaIndex 에이전트를 자동으로 계측합니다.

이 배포판은 다음 기능을 제공합니다.
- AgentCore Runtime 외부(예: EC2, Lambda 등)에서 호스팅되는 LlamaIndex Agent를 위한 **자동 계측**
- 원활한 CloudWatch 통합을 위한 **AWS 최적화 구성**

### 계측된 에이전트 실행

LlamaIndex 에이전트의 trace를 수집하려면 Python을 직접 실행하는 대신 `opentelemetry-instrument` 명령을 사용합니다. 이 명령은 `.env` 파일의 environment variable을 사용하여 자동으로 계측을 적용합니다.

```bash
opentelemetry-instrument python llama_index_agent.py
```

이 명령은 다음 작업을 수행합니다.

- .env 파일에서 OTEL 구성 불러오기
- LlamaIndex, Amazon Bedrock 호출, 에이전트 tool과 database 및 에이전트가 보내는 기타 request를 자동으로 계측
- CloudWatch로 trace 전송
- GenAI Observability dashboard에서 에이전트의 의사 결정 과정을 시각화

In [ ]:
!opentelemetry-instrument python llama_index_agent.py

## 6. Session Tracking 추가

여러 에이전트 실행의 trace를 연결하려면 OpenTelemetry baggage를 사용하여 telemetry 데이터에 session ID를 연결할 수 있습니다.

```python
from opentelemetry import baggage, context
ctx = baggage.set_baggage("session.id", session_id)
```

session이 활성화된 버전을 실행합니다.
```bash
opentelemetry-instrument python llama_indedx_agent_with_session.py --session-id "user-session-123"
```

## 7. 분석용 Custom Metadata
필터링, offline evaluation 및 성능 분석을 위해 custom attribute를 추가합니다. 추가 parameter를 받도록 에이전트 코드를 수정해야 합니다.
```python
ctx = baggage.set_baggage("user.type", "premium")
ctx = baggage.set_baggage("experiment.id", "llama-agent")
ctx = baggage.set_baggage("conversation.topic", "arithmetic")
```

custom metadata를 사용하는 명령 예제:

```bash
# 서로 다른 실험 A/B 테스트
opentelemetry-instrument python agent.py --session-id "session-123" --experiment-id "model-a"
opentelemetry-instrument python agent.py --session-id "session-124" --experiment-id "model-b"

# 서로 다른 사용자 유형 추적
opentelemetry-instrument python agent.py --session-id "session-125" --user-type "premium"
opentelemetry-instrument python agent.py --session-id "session-126" --user-type "free"

# Offline evaluation 실행
opentelemetry-instrument python agent.py --session-id "eval-001" --dataset "golden-set-v1"
```
이러한 attribute는 고급 필터링과 분석을 위해 CloudWatch trace에 표시됩니다.

In [ ]:
%%writefile llama_index_agent_with_session.py
import os
import logging
import argparse
import asyncio
from opentelemetry import baggage, context
from llama_index.observability.otel import LlamaIndexOpenTelemetry
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent

def parse_arguments():
    parser = argparse.ArgumentParser(description='LlamaIndex Arithmetic Agent with Session Tracking')
    parser.add_argument('--session-id', 
                       type=str, 
                       required=True,
                       help='Session ID to associate with this agent run')
    return parser.parse_args()

def set_session_context(session_id):
    """트레이스 연계를 위해 OpenTelemetry Baggage에 세션 ID를 설정합니다."""
    ctx = baggage.set_baggage("session.id", session_id)
    token = context.attach(ctx)
    logging.info(f"Session ID '{session_id}' attached to telemetry context")
    return token

###########################
#### 아래는 Agent 코드입니다. ####
###########################

# LlamaIndex용 OpenTelemetry 계측 초기화
instrumentor = LlamaIndexOpenTelemetry(debug=True)

# 로깅 설정
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# LlamaIndex 로깅 구성
logging.getLogger("llamaindex").setLevel(logging.INFO)

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b

def get_bedrock_model():
    model_id = os.getenv("BEDROCK_MODEL_ID", "global.anthropic.claude-haiku-4-5-20251001-v1:0")
    region = os.getenv("AWS_DEFAULT_REGION", "us-west-2")

    try:
        # boto3가 credential resolution을 자동으로 처리하도록 설정
        bedrock_model = BedrockConverse(
            model=model_id,
            region_name=region,
            # credential을 명시하지 않아도 boto3가 자동으로 검색
        )
        logger.info(f"Successfully initialized Bedrock model: {model_id} in region: {region}")
        return bedrock_model
    except Exception as e:
        logger.error(f"Failed to initialize Bedrock model: {str(e)}")
        logger.error("Please ensure you have proper AWS credentials configured and access to the Bedrock model")
        raise

async def run_agent(query):
    # 모델 초기화
    bedrock_model = get_bedrock_model()

    # 산술 에이전트 생성
    agent = FunctionAgent(
        tools=[add, multiply],
        llm=bedrock_model,
    )

    # 수신 시작
    instrumentor.start_registering()

    # 산술 task 실행
    result = await agent.run(query)
    print("Result:", str(result))
    return result

def main():
    # command line argument 구문 분석
    args = parse_arguments()

    # telemetry용 session context 설정
    context_token = set_session_context(args.session_id)

    try:
        # 산술 task 실행
        query = """What is (121 + 2) * 5?"""

        # event loop에서 비동기 함수 실행
        result = asyncio.run(run_agent(query))

    finally:
        # 완료 후 context 분리
        try:
            context.detach(context_token)
            logger.info(f"Session context for '{args.session_id}' detached")
        except ValueError as e:
            # 발생할 수 있는 context 분리 오류 처리
            logger.error(f"Error detaching context: {str(e)}")

if __name__ == "__main__":
    main()



In [ ]:
!opentelemetry-instrument python llama_index_agent_with_session.py --session-id "session-1234"

## 8. Gen AI Observability Dashboard에서 AWS CloudWatch Trace 이해하기

LlamaIndex 에이전트를 OpenTelemetry 계측과 함께 실행하면 AWS CloudWatch의 GenAI Observability dashboard에서 trace를 시각화하고 분석할 수 있습니다. Bedrock AgentCore로 이동하여 방금 생성한 Agent를 클릭합니다.

#### Sessions View 페이지:

![llama_index_sessions.png](images/llama_index_sessions.png)


#### Trace View 페이지:
Trace View:

![llama_index_sessions.png](images/llama_index_traces.png)


Trace 세부 정보:

![llama_index_sessions.png](images/llama_index_trace_details.png)



## 9. 문제 해결

Amazon CloudWatch 또는 X-Ray에서 trace가 보이지 않으면 다음 항목을 확인하세요.

1. **AWS Credentials**: AWS credentials가 올바르게 구성되어 있는지 확인합니다.
2. **IAM Permissions**: IAM user/role에 CloudWatch 권한이 있는지 확인합니다.
3. **Region**: 올바른 AWS Region을 확인하고 있는지 점검합니다.
4. **Environment Variables**: 모든 OTEL_* environment variable이 올바르게 설정되어 있는지 확인합니다.

## 10. 마무리

축하합니다. Amazon Bedrock 모델을 사용하는 LlamaIndex Agent를 구현하고 계측하여 Amazon CloudWatch를 통한 관찰성을 구성했습니다.

- LlamaIndex 산술 에이전트
- 전체 OpenTelemetry tracing
- Amazon Bedrock 호출, LlamaIndex 작업 등에 대한 trace
- 서비스 이름: agentic-llamaindex-agentcore

## 11. 다음 단계

이제 LlamaIndex와 OpenTelemetry 설정을 마쳤으므로 다음 작업을 수행할 수 있습니다.

1. **에이전트 추가**: 서로 다른 pattern을 사용하는 multi-agent architecture 생성
2. **에이전트에 Tool 추가**: 검색 tool, API tool 또는 custom tool 통합
3. **[Alarm 설정](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/AlarmThatSendsEmail.html)**: `latency`, `token input`, `token output` 등 비즈니스에 중요한 metric에 alarm 생성
